In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.8 Exact Conditions and the Band-Gap Problem

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.8",
    title="Exact Conditions and the Band-Gap Problem",
    blurb="Movement II closes on density-functional theory's deepest structure: what "
    "the exact functional must do at fractional electron number, and what the "
    "local approximation does instead. Janak's theorem verified at machine "
    "precision, the exact energy's piecewise-linear walk between integers laid "
    "over the LDA's smooth convex lie, the derivative discontinuity extracted as "
    "a number, the band-gap problem assembled from first principles on a system "
    "whose every ingredient is exactly known — and one repair, the "
    "self-interaction correction, restoring a missing tenth of a Hartree to "
    "within twenty micro-Hartree.",
    difficulty="advanced",
    estimate="130–160 min",
)

## Notebook overview

Every failure catalogued in [§8.7](kohn-sham.ipynb) — eigenvalues too high,
potentials too shallow, hydrogen repelling itself — turns out to be one failure
wearing different clothes, and this notebook undresses it. The deep question is
what the energy does as the electron *number* varies, integers and fractions
alike. Perdew, Parr, Levy, and Balduz {cite}`pplb1982` answered it exactly: the
exact $E(N)$ is a sequence of straight-line segments between the integers, with
derivative jumps *at* them — and the sizes of those jumps are ionization
energies, electron affinities, and, in solids, the band gap itself. The local
density approximation replaces the segments with a smooth convex curve, and
every frontier pathology follows from that single geometric mistake.

The laboratory of [§8.2](exact-laboratory.ipynb) makes all of it computable.
Janak's theorem ($\partial E/\partial f_i = \varepsilon_i$) is verified at
machine precision on the from-scratch LDA of [§8.7](kohn-sham.ipynb) run at
genuinely fractional electron number. The exact $E(N)$ ladder is assembled from
the laboratory's certificate ($E(0), E(1), E(2)$ all exactly known) and its
piecewise-linear ensemble interpolation stated with the PPLB argument; the LDA's
$E(N)$ is *computed* at thirteen fractional numbers and its convexity measured
($0.08$ Ha below the exact chord at $N = 3/2$). At $N = 1$ the gap anatomy is
exact and complete: fundamental gap $I - A = 0.7288$ Ha, Kohn–Sham gap $0.7111$
Ha (the exact KS potential of one electron is the bare well — a rare case where
the exact functional's gap is *known*), and their difference, the derivative
discontinuity $\Delta_{xc} = 0.0177$ Ha, extracted as a plain number. The
frontier-slope analysis then assembles the band-gap problem from first
principles, and the notebook closes with the first repair: the Perdew–Zunger
self-interaction correction {cite}`perdewzunger1981`, which returns the
one-electron laboratory to the exact energy within $2\times10^{-5}$ Ha.

> **Conventions (this notebook).** Hartree atomic units; the
> [§8.2](exact-laboratory.ipynb) laboratory grid and interaction throughout,
> with the [§8.7](kohn-sham.ipynb) from-scratch exchange-only LDA (its
> uniform-gas $\varepsilon_x$ spline rebuilt in Setup so the notebook runs
> standalone). "LDA at fractional $N$" means the single KS orbital occupied by
> $N$ electrons, $n = N|\varphi|^2$, self-consistently. The exact integer
> energies are the laboratory certificate values; ensemble fractional values
> are the PPLB linear interpolation *by theorem*, and are labeled as such.
>
> **How to read the checks.** Each exercise closes with a `validate` call
> against an independent fact: a derivative identity, certificate energies, a
> measured convexity, an exact gap decomposition. A ✓ is strong evidence; a ✗
> is a prompt to locate the discrepancy, not an automatic verdict.
>
> **Scope.** The PPLB theorem is stated with its ensemble argument and its
> consequences computed; the grand-canonical derivation in full lives in
> {cite}`pplb1982` and Parr & Yang {cite}`parryang1989`, Ch. 4. Janak's
> original proof is {cite}`janak1978`. The band-gap discussion for real solids
> (where quasiparticle methods take over) is the business of
> [§8.14](quasiparticles-gw.ipynb); here the problem is *assembled*, exactly,
> on a system with no experimental error bars.

## Theory in brief

### Janak: what a Kohn–Sham eigenvalue is

Kohn–Sham eigenvalues entered as Lagrange multipliers, and
[§8.3](hartree-fock-atoms.ipynb)'s Koopmans theorem gave Hartree–Fock's version
of their meaning. Janak {cite}`janak1978` gave the density-functional version,
and it is exact for *any* functional, approximate ones included: allow the
occupation $f_i$ of orbital $i$ to vary continuously, and

```{math}
:label: eq-gap-janak
\frac{\partial E}{\partial f_i} = \varepsilon_i
```

— the eigenvalue is the marginal price of occupation. The proof is three lines
of the [§8.5](thomas-fermi.ipynb) functional calculus: differentiating
$E[\{f_j\}]$ through the self-consistent density, every implicit term cancels
by stationarity, leaving the explicit one, $\varepsilon_i$. The theorem makes
eigenvalues *thermodynamic* objects (slopes of $E$ versus particle number), and
that is exactly the language the main theorem speaks.

### PPLB: the exact energy walks in straight lines

What is $E(N)$ for fractional $N$? Fractional charge is no fiction — a system
exchanging electrons with a distant reservoir (a dissociating molecule, an atom
far from a surface) holds one on *average* — and the correct energy comes from
minimizing over ensembles. Perdew, Parr, Levy, and Balduz {cite}`pplb1982`
showed the minimizing ensemble at $N = M + \omega$ (integer $M$, fraction
$\omega$) simply mixes the two adjacent integer ground states, so

```{math}
:label: eq-gap-pplb
E(M + \omega) = (1 - \omega)\,E(M) + \omega\,E(M+1):
```

straight segments between integers, with slopes $E(M{+}1) - E(M) = -A_M$
(minus the affinity) on the right of each integer and $-I_M$ (minus the
ionization energy) on the left. At each integer the slope *jumps* by
$I - A$ — the fundamental gap — and since Janak ties slopes to frontier
eigenvalues, the exact Kohn–Sham potential must jump by a constant,
the **derivative discontinuity** $\Delta_{xc}$, as $N$ crosses an integer:

```{math}
:label: eq-gap-anatomy
E_{\mathrm{gap}} \;=\; I - A
\;=\; \underbrace{\varepsilon_{\mathrm{LUMO}} - \varepsilon_{\mathrm{HOMO}}}_{\text{KS gap}}
\;+\; \Delta_{xc} .
```

Even the *exact* Kohn–Sham gap undershoots the fundamental gap; the missing
piece is a property of the functional, not of the eigenvalue spectrum. Any
approximation with a smooth $E(N)$ — the LDA above all — has $\Delta_{xc} = 0$
*and* misplaced slopes, and its convex sag between integers is the
delocalization error that plagues dissociation curves, charge transfer, and
gaps across computational chemistry and materials science.

### The one-electron stage

Every quantity in Eq. {eq}`eq-gap-anatomy` is exactly computable on the
laboratory at $N = 1$. The integer energies $E(0) = 0$,
$E(1) = -1.483663$, $E(2) = -2.238550$ Ha are certificate values from
[§8.2](exact-laboratory.ipynb); they give $I = 1.483663$ and $A = 0.754887$
Ha. And the exact Kohn–Sham system of *one* electron is a rare open book: with
nothing to interact with, its exact KS potential is the bare well itself, so
the exact KS gap is just the well's level spacing — leaving $\Delta_{xc}$ as
an honest subtraction. No system with experimental error bars allows that.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from scipy.linalg import eigh_tridiagonal

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

# The laboratory grid and certificate (section 8.2), restated for standalone runs
x = np.linspace(-10.0, 10.0, 201)
h = x[1] - x[0]
W_INT = 1.0 / np.sqrt((x[:, None] - x[None, :]) ** 2 + 1.0)
V_EXT = -2.0 / np.sqrt(x**2 + 1.0)
OFF = np.full(200, -0.5 / h**2)
E_CERT = {0: 0.0, 1: -1.483663, 2: -2.238550}  # exact integer energies

# The 8.7 from-scratch uniform-gas exchange for the laboratory's interaction,
# rebuilt as a spline so this notebook runs standalone.


def _eps_x_gas(n_1d):
    """Exchange per electron of the uniform 1-D soft-Coulomb gas (section 8.7)."""
    if n_1d < 1e-10:
        return 0.0
    k_f = np.pi * n_1d / 2.0
    val, _ = quad(
        lambda u: (np.sin(k_f * u) / (np.pi * u)) ** 2 / np.sqrt(u * u + 1.0),
        1e-9,
        300.0,
        limit=500,
    )
    return -2.0 * val / n_1d


_n_tab = np.linspace(1e-4, 1.6, 160)
EX_SPLINE = CubicSpline(_n_tab, _n_tab * np.array([_eps_x_gas(v) for v in _n_tab]))
V_X_LDA = EX_SPLINE.derivative()


def lda_lab(n_electrons, tol=1e-10, max_iter=600):
    """Self-consistent exchange-only LDA for the laboratory at (fractional) N.

    One Kohn-Sham orbital carries all N electrons, n = N |phi|^2 — the
    fractional-occupation setting Janak's theorem speaks about. The loop is
    the damped fixed point of section 8.7 with mixing 1/2.

    Parameters
    ----------
    n_electrons : float
        Electron number, integer or fractional.
    tol : float, optional
        Density convergence threshold.
    max_iter : int, optional
        Iteration cap.

    Returns
    -------
    tuple
        ``(E, eps, n)``: total energy (Hartree), the occupied orbital
        eigenvalue, and the converged density.
    """
    dens = np.exp(-(x**2))
    dens *= n_electrons / np.trapezoid(dens, x)
    for _ in range(max_iter):
        v_h = h * (W_INT @ dens)
        v_x = V_X_LDA(np.clip(dens, 0.0, None))
        eps, u = eigh_tridiagonal(
            1.0 / h**2 + V_EXT + v_h + v_x, OFF, select="i", select_range=(0, 0)
        )
        phi = u[:, 0] / np.sqrt(h)
        dens_new = n_electrons * phi**2
        if np.abs(dens_new - dens).max() < tol:
            break
        dens = 0.5 * dens + 0.5 * dens_new
    h_phi = (1.0 / h**2) * phi + V_EXT * phi
    h_phi[:-1] += OFF * phi[1:]
    h_phi[1:] += OFF * phi[:-1]
    energy = (
        n_electrons * np.sum(phi * h_phi) * h
        + 0.5 * h * np.trapezoid(dens * (W_INT @ dens), x)
        + np.trapezoid(EX_SPLINE(np.clip(dens, 0.0, None)), x)
    )
    return energy, eps[0], dens

## Exercise 1 — Janak, at machine precision

Equation {eq}`eq-gap-janak` holds for any smooth functional, so the
[§8.7](kohn-sham.ipynb) LDA — which can occupy its orbital fractionally with
no ceremony — is the perfect test bench. The theorem's content is that the
*total-energy* slope, with all the self-consistent rearrangement it entails,
collapses to the bare eigenvalue.

**Part a)** At $N = 3/2$ (deliberately far from any integer), compute
$\partial E/\partial N$ by the central difference of two full SCF solutions at
$N = 1.5 \pm 0.01$ (`numpy` arithmetic on the Setup helper `lda_lab`), and
compare with the eigenvalue $\varepsilon(N{=}1.5)$ from a third solution.

**Part b)** Sweep $N$ from $0.4$ to $2.0$ in steps of $0.2$: at every point the
finite-difference slope must ride the eigenvalue curve. Plot both; the
eigenvalue *is* the slope, everywhere.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the marginal price of occupation

The central-difference slope at $N = 3/2$ must match the eigenvalue to
$10^{-4}$ (the finite-difference step's own quadratic error), and the sweep's
maximum deviation must stay below $10^{-3}$.

In [ ]:
validate.close(slope_fd, eps_mid, "Janak at N = 3/2", rtol=1e-4)
validate.close(
    float(np.abs(slopes_sweep - eps_sweep).max()),
    0.0,
    "Janak across the whole sweep",
    rtol=0.0,
    atol=1e-3,
)

## Exercise 2 — The exact energy walks in straight lines

The laboratory's certificate holds all three integer energies, and the PPLB
theorem of Eq. {eq}`eq-gap-pplb` dictates everything between them. The
fractional values below are the theorem's ensemble interpolation — labeled as
constructed, since the *content* to verify is not the segments (they are
straight by theorem) but the slopes and the kink.

**Part a)** Assemble the exact $E(N)$ on $N \in [0, 2]$ from the certificate
via Eq. {eq}`eq-gap-pplb` (`numpy.interp` over the three integer anchors), and
extract the two slopes at $N = 1$ by differencing the segments: the left slope
must equal $-I = -1.483663$ and the right slope $-A = -0.754887$ Ha — energies
to remove or add an electron, read off a graph's kink.

**Part b)** The jump of the slope at $N = 1$ is the one-electron system's
fundamental gap $I - A$. Report it: $0.728776$ Ha, a number Exercise 4 will
dissect.

In [ ]:
# (solution hidden on the public site)


### Validation 2 — ionization and affinity, read off a kink

The slopes must reproduce the certificate's $I$ and $A$, and the jump their
difference: thermodynamics from geometry.

In [ ]:
validate.close(I_ONE, 1.483663, "the ionization energy as the left slope", rtol=1e-6)
validate.close(A_ONE, 0.754887, "the electron affinity as the right slope", rtol=1e-6)
validate.close(
    gap_fundamental, 0.728776, "the fundamental gap as the slope jump", rtol=1e-5
)

## Exercise 3 — The LDA's smooth convex lie

Now the same walk in the approximate world. The LDA has no ensemble subtlety:
its $E(N)$ comes from simply *running the SCF* at fractional occupation, and
the result is a smooth convex curve — no kink, no straight segments, and a sag
below the exact chord that *is* the delocalization error, the single geometric
fact behind fractional charges on dissociating molecules and underestimated
gaps in solids.

**Part a)** Compute the LDA $E(N)$ at the thirteen numbers
$N = 0.2, 0.35, \dots, 2.0$ (`numpy.linspace`, the Setup helper), and measure
the maximum sag below the exact-functional chord between $N = 1$ and $2$ (the
chord of the *LDA's own* integer endpoints, so the comparison isolates
curvature from the integer-point errors).

**Part b)** Overlay the exact piecewise-linear walk and the LDA curve: the
volume's single most consequential figure. Note both diseases at once: the
*curvature* (no kink anywhere, sag $0.081$ Ha at $N = 3/2$) and the *integer
error* ($E_{\mathrm{LDA}}(1) = -1.3713$ vs the exact $-1.4837$: one electron's
self-interaction, about to be cured in Exercise 6).

In [ ]:
# (solution hidden on the public site)


### Validation 3 — the two diseases, measured

The LDA integer values must land at their measured $-1.3713$ and $-2.1584$ Ha;
the sag must be the measured $-0.081$ Ha (strictly convex everywhere: second
differences positive); and the $N = 1$ self-interaction error must be
$0.112$ Ha.

In [ ]:
validate.close(E_lda_1, -1.37130, "the LDA one-electron energy", rtol=1e-3)
validate.close(E_lda_2, -2.15845, "the LDA two-electron energy", rtol=1e-3)
validate.close(sag, -0.0806, "the delocalization sag at N = 3/2", rtol=2e-2)
validate.check(
    bool(np.all(np.diff(E_lda_N, 2) > 0.0)),
    "the LDA E(N) is strictly convex (no kink anywhere)",
    "second differences positive at every interior point",
)
validate.close(
    E_lda_1 - E_CERT[1], 0.11236, "one electron's self-interaction error", rtol=2e-2
)

## Exercise 4 — The gap anatomy, exactly

Equation {eq}`eq-gap-anatomy` decomposes the fundamental gap into a Kohn–Sham
part and a functional part, and at $N = 1$ the laboratory computes *both
exactly*: the fundamental gap came from Exercise 2, and the exact KS system of
one electron is the bare well itself (nothing to interact with, so
$v_H + v_{xc} = 0$ for the exact functional), making the exact KS gap the
well's plain level spacing.

**Part a)** Diagonalize the bare well with `scipy.linalg.eigh_tridiagonal`
(two lowest states) and form the exact KS gap
$\varepsilon_1 - \varepsilon_0 = 0.7111$ Ha.

**Part b)** Subtract per Eq. {eq}`eq-gap-anatomy`:
$\Delta_{xc} = (I - A) - (\varepsilon_1 - \varepsilon_0) = 0.0177$ Ha. Plot
the decomposition as a stacked bar. The moral is precise: even with the
*exact* functional, the KS eigenvalue gap undershoots the fundamental gap by
a finite $\Delta_{xc}$ — the missing piece is a jump of the potential, not an
eigenvalue error — and any smooth-functional gap (LDA's included) misses both
this piece *and* carries misplaced eigenvalues on top.

In [ ]:
# (solution hidden on the public site)


### Validation 4 — the discontinuity is real and finite

The exact KS gap must be the measured $0.711104$ Ha, and $\Delta_{xc}$ the
measured $0.017672$ Ha — positive, finite, and $2.4\%$ of the gap here
(in real semiconductors it can reach tens of percent).

In [ ]:
validate.close(
    gap_ks, 0.711104, "the exact KS gap of the one-electron system", rtol=1e-5
)
validate.close(delta_xc, 0.017672, "the derivative discontinuity", rtol=1e-3)
validate.check(
    delta_xc > 0.0,
    "Delta_xc is positive",
    "the potential jumps upward as N crosses the integer",
)

## Exercise 5 — The band-gap problem, assembled

The pieces now assemble into the statement every electronic-structure
practitioner carries: *Kohn–Sham gaps underestimate fundamental gaps, and
approximate functionals make it worse.* The two layers, kept distinct:

1. **The functional's layer.** Even exact, the KS gap misses $\Delta_{xc}$
   (Exercise 4). This is structural — no eigenvalue spectrum contains it.
2. **The approximation's layer.** A smooth $E(N)$ has no derivative
   discontinuity at all *and* misplaces its frontier slopes: by Janak, the
   LDA's slope as $N \to 2^-$ is its HOMO eigenvalue, which
   [§8.7](kohn-sham.ipynb) measured $38\%$ shy of $-I$.

**Part a)** Extract the LDA's frontier slope at $N \to 2^-$ from the Exercise
3 data (the last finite difference of `numpy.diff` over the final segment) and
compare with the exact slope $-I(2) = -0.754887$ Ha (the certificate's
two-electron ionization energy): the LDA "HOMO level" sits high by the
measured $\approx 0.25$ Ha.

**Part b)** Tabulate the full indictment for the laboratory at $N = 1$: the
fundamental gap ($0.7288$), the exact KS gap ($0.7111$), and the LDA gap
proxy $\varepsilon_1 - \varepsilon_0$ of the LDA potential at $N = 1$
(diagonalize the converged $N = 1$ LDA Hamiltonian for two states with
`scipy.linalg.eigh_tridiagonal`) — measured at $0.70$ Ha, under the exact KS
gap, under the fundamental gap: the two layers stacking, on a system with no
experimental error bars.

In [ ]:
# (solution hidden on the public site)


### Validation 5 — two layers, both measured

The LDA frontier slope must sit above the exact $-I$ by $0.2$–$0.3$ Ha (the
self-interaction lift), and the gap ladder must order strictly:
fundamental > exact KS > LDA.

In [ ]:
validate.check(
    0.2 < slope_lda_frontier + 0.754887 < 0.3,
    "the LDA frontier slope sits high by the self-interaction lift",
    f"{slope_lda_frontier + 0.754887:+.3f} Ha above -I",
)
validate.check(
    gap_fundamental > gap_ks > gap_lda,
    "the gap ladder: fundamental > exact KS > LDA",
    f"{gap_fundamental:.4f} > {gap_ks:.4f} > {gap_lda:.4f}",
)

## Exercise 6 — The first repair: self-interaction correction

Movement II closes constructively. Perdew and Zunger's 1981 appendix
{cite}`perdewzunger1981` proposed the obvious surgery: subtract, orbital by
orbital, the spurious self-terms —

```{math}
:label: eq-gap-sic
E^{\mathrm{SIC}} = E^{\mathrm{LDA}}
- \sum_{i}^{\mathrm{occ}} \Big( E_H[n_i] + E_{xc}^{\mathrm{LDA}}[n_i] \Big),
```

which for *one* electron removes the Hartree self-repulsion and the spurious
self-exchange exactly, leaving the bare $\langle\varphi|\hat h|\varphi\rangle$.
The correction is evaluated here non-self-consistently (on the converged LDA
orbital), which costs only the orbital's tiny residual relaxation.

**Part a)** For the laboratory at $N = 1$, evaluate Eq. {eq}`eq-gap-sic`:
subtract $E_H[n_1]$ (the half-kernel double integral, `numpy.trapezoid`) and
$E_x^{\mathrm{LDA}}[n_1]$ (the exchange spline) from the Exercise 3 LDA
energy.

**Part b)** Compare with the exact $E(1) = -1.483663$: the correction closes
the $0.112$-Ha error to $2\times10^{-5}$ Ha (the unrelaxed orbital's
variational residual). One line of physics recovers four digits — and the
honest coda: for *many* orbitals the SIC functional becomes
orbital-dependent, losing the KS structure's simplicity, which is why the
modern road runs instead through hybrids and the many-body methods of
Movement IV.

In [ ]:
# (solution hidden on the public site)


### Validation 6 — four digits from one line

The corrected energy must land within $10^{-4}$ Ha of the exact one-electron
energy (measured residual $2\times10^{-5}$, the unrelaxed orbital's
second-order price), repairing a $0.112$-Ha disease.

In [ ]:
validate.close(
    E_sic,
    E_CERT[1],
    "the self-interaction-corrected one-electron energy",
    rtol=0.0,
    atol=1e-4,
)
validate.check(
    abs(residual_sic) < 1e-4 < 0.112,
    "a 0.112-Ha disease repaired to 2e-5",
    f"residual {residual_sic:+.1e} Ha",
)

:::{admonition} With your assistant
:class: tip
The delocalization sag of Exercise 3 has a famous molecular consequence:
stretched H$_2^+$ dissociating to two half-charges. Have your assistant adapt
the fractional-$N$ LDA to the two-center laboratory potential of
[§8.1](many-electron-problem.ipynb) at large separation and compute
$E(\text{molecule}, N{=}1)$ against $2\times E(\text{atom}, N{=}1/2)$, then
run the check that is yours alone: the LDA must *prefer* the two half-charges
(the fractional configuration lower, by the convexity you measured), which is
exactly backwards — the exact piecewise-linear functional is indifferent —
and the sign of that preference (`numpy` comparison) is the entire
delocalization-error story in one inequality. The check is yours.
:::

## Notebook summary

Movement II closes with density-functional theory's deepest structure
computed rather than recited. Janak's theorem held at machine sharpness
($\partial E/\partial N = \varepsilon$ to $10^{-5}$ at $N = 3/2$ and $10^{-3}$
across the sweep). The exact $E(N)$ walked its PPLB straight lines through the
certificate anchors, its kink at $N = 1$ pricing $I = 1.4837$ and
$A = 0.7549$ Ha; the LDA's $E(N)$, computed at thirteen fractional numbers,
was strictly convex with a $0.081$-Ha delocalization sag and a $0.112$-Ha
one-electron self-interaction error. The gap anatomy came out exact:
fundamental $0.7288 =$ exact-KS $0.7111 + \Delta_{xc}\,0.0177$ Ha, and the
assembled band-gap problem ordered its ladder — fundamental $>$ exact KS $>$
LDA ($0.70$) — with the LDA frontier slope high by $0.24$ Ha, the
[§8.7](kohn-sham.ipynb) IP failure in $E(N)$ language. The Perdew–Zunger
self-interaction correction then repaired the one-electron energy from
$-1.3713$ to $-1.48364$ against the exact $-1.483663$: four digits from one
line, and Movement II's honest note that the many-orbital version trades away
the very simplicity that makes Kohn–Sham run the world.

## Outlook

- Movement III changes arena: electrons in *crystals*, where the KS machinery
  meets Bloch's theorem and the band structures of
  [§7.12](../07-quantum-statistical-mechanics/bloch-theorem-band-structure.ipynb)
  become computable for real materials
  ([§8.9](tight-binding.ipynb)–[§8.12](berry-wannier-ssh.ipynb)). The gap
  problem assembled here travels with them.
- The honest gap machinery — quasiparticles, self-energies, the GW
  approximation that repairs $\Delta_{xc}$'s absence from the outside — is
  Movement IV's opening business ([§8.14](quasiparticles-gw.ipynb)), with the
  thesis-grade worked example promised there.
- Piecewise linearity has become a design principle: modern "curvature-free"
  and range-separated functionals are tuned to walk straight between
  integers. The laboratory built here is exactly the kind of test bench on
  which such functionals are certified.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()